In [9]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path(r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump")

INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "financial" / "crop_soil_climate_with_revenue_success_2013_2025.csv"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "modeling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELED_FILE = OUT_DIR / "model_ready_labeled_2013_2025.csv"
UNLABELED_FILE = OUT_DIR / "model_unlabeled_2013_2025.csv"
TRAIN_FILE = OUT_DIR / "model_train_2013_2022.csv"
VALIDATION_FILE = OUT_DIR / "model_validation_2023.csv"
TEST_FILE = OUT_DIR / "model_test_2024.csv"
FEATURE_DICT_FILE = OUT_DIR / "model_feature_dictionary.csv"
AUDIT_FILE = OUT_DIR / "modeling_quality_audit.csv"
SUMMARY_FILE = OUT_DIR / "FINAL_MODEL_DATASET_HANDOFF.txt"
JSON_SUMMARY_FILE = OUT_DIR / "final_model_dataset_summary.json"

print("Input:", INPUT_FILE)
print("Output directory:", OUT_DIR)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Notebook 15 output not found: {INPUT_FILE}")


Input: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\financial\crop_soil_climate_with_revenue_success_2013_2025.csv
Output directory: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\modeling


In [10]:
df = pd.read_csv(INPUT_FILE)
print("Shape:", df.shape)
print("Columns:", len(df.columns))
display(pd.DataFrame({"column": df.columns}))


Shape: (67826, 67)
Columns: 67


,column
0,year
1,state
2,district
3,state_code
4,district_code
...,...
62,actual_revenue_rs_per_acre
63,target_revenue_rs_per_acre
64,revenue_vs_target_ratio
65,success_label


In [11]:
#Hard structural checks
EXPECTED_ROWS = 67826
EXPECTED_YEARS = set(range(2013, 2025))
EXPECTED_CROPS = {
    "Arhar/Tur", "Bajra", "Gram", "Groundnut", "Jowar", "Maize",
    "Ragi", "Rice", "Soyabean", "Sugarcane", "Urad", "Wheat"
}
CORE_KEY = ["state", "district", "crop", "soil_type", "year"]

required_columns = [
    "year", "state", "district", "crop", "season", "soil_type",
    "area_ha", "yield_kg_ha", "production_tonnes",
    "historical_10yr_baseline_yield_kg_ha",
    "historical_baseline_year_count",
    "historical_baseline_start_year",
    "historical_baseline_end_year",
    "historical_baseline_yield_q_acre",
    "support_price_rs_per_quintal",
    "support_price_type",
    "yield_q_acre",
    "actual_revenue_rs_per_acre",
    "target_revenue_rs_per_acre",
    "revenue_vs_target_ratio",
    "success_label",
]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required Notebook 15 columns: {missing_required}")

df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
df["success_label"] = pd.to_numeric(df["success_label"], errors="coerce")

assert len(df) == EXPECTED_ROWS
assert set(df["year"].unique()) == EXPECTED_YEARS
assert set(df["crop"].dropna().unique()) == EXPECTED_CROPS
assert df.duplicated(CORE_KEY).sum() == 0

print("PASS: 67,826 rows")
print("PASS: crop years 2013–2024")
print("PASS: 12 crops")
print("PASS: duplicate core records = 0")


PASS: 67,826 rows
PASS: crop years 2013–2024
PASS: 12 crops
PASS: duplicate core records = 0


In [12]:
#Target availability
label_available = df["success_label"].isin([0, 1])
labeled = df.loc[label_available].copy()
unlabeled = df.loc[~label_available].copy()

print("Total:", len(df))
print("Labeled:", len(labeled))
print("Unlabeled:", len(unlabeled))

class_balance = labeled["success_label"].value_counts().sort_index().to_frame("rows")
class_balance["percent"] = class_balance["rows"] / len(labeled) * 100
display(class_balance)

if "financial_label_status" in df.columns:
    display(df["financial_label_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))

assert len(labeled) == 58185
assert len(unlabeled) == 9641
assert set(labeled["success_label"].unique()) == {0.0, 1.0}


Total: 67826
Labeled: 58185
Unlabeled: 9641


,rows,percent
success_label,,
0.0,22307,38.33806
1.0,35878,61.66194


,status,rows
0,label_available,58185
1,missing_historical_baseline,7217
2,missing_actual_yield,2355
3,invalid_target,69


In [13]:
#Leakage-safe model feature set
categorical_features = [
    "state", "district", "crop", "season", "soil_type", "support_price_type"
]

numeric_features = [
    "year", "area_ha",
    "historical_10yr_baseline_yield_kg_ha",
    "historical_baseline_year_count",
    "historical_baseline_start_year",
    "historical_baseline_end_year",
    "historical_baseline_yield_q_acre",
    "support_price_rs_per_quintal",
    "clayey_fraction", "clayey_skeletal_fraction", "loamy_fraction", "sandy_fraction",
    "annual_rainfall_mm", "annual_mean_temp_c", "annual_max_temp_c",
    "annual_min_temp_c", "annual_relative_humidity_pct", "annual_wind_speed_m_s",
    "annual_solar_radiation", "monsoon_rainfall_mm", "monsoon_mean_temp_c",
    "monsoon_max_temp_c", "monsoon_min_temp_c", "monsoon_relative_humidity_pct",
    "monsoon_wind_speed_m_s", "monsoon_solar_radiation"
]

FEATURES = categorical_features + numeric_features
TARGET = "success_label"

missing_features = [c for c in FEATURES + [TARGET] if c not in labeled.columns]
if missing_features:
    raise ValueError(f"Missing model columns: {missing_features}")

LEAKAGE_EXCLUSIONS = [
    "yield_kg_ha", "yield_q_acre", "production_tonnes",
    "actual_revenue_rs_per_acre", "revenue_vs_target_ratio",
    "yield_vs_historical_baseline_ratio", "target_revenue_rs_per_acre",
    "success_label"
]
assert not any(c in FEATURES for c in LEAKAGE_EXCLUSIONS)

print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))
print("Total features:", len(FEATURES))


Categorical features: 6
Numeric features: 26
Total features: 32


In [14]:
#Feature completeness and physical sanity checks
missing_audit = labeled[FEATURES].isna().sum().sort_values(ascending=False)
display(missing_audit[missing_audit > 0])

if missing_audit.sum() != 0:
    raise ValueError("Selected model features contain missing values.")

nonnegative_columns = [
    "area_ha", "historical_10yr_baseline_yield_kg_ha",
    "historical_baseline_year_count", "historical_baseline_yield_q_acre",
    "support_price_rs_per_quintal", "clayey_fraction",
    "clayey_skeletal_fraction", "loamy_fraction", "sandy_fraction",
    "annual_rainfall_mm", "monsoon_rainfall_mm"
]
for col in nonnegative_columns:
    assert (labeled[col] >= 0).all(), f"Negative value in {col}"

for col in ["clayey_fraction", "clayey_skeletal_fraction", "loamy_fraction", "sandy_fraction"]:
    assert labeled[col].between(0, 1).all(), f"Texture fraction outside [0,1]: {col}"

assert labeled["historical_baseline_year_count"].between(0, 10).all()
assert labeled["support_price_rs_per_quintal"].gt(0).all()

print("PASS: zero missing model-feature cells")
print("PASS: physical ranges valid")


Series([], dtype: int64)

PASS: zero missing model-feature cells
PASS: physical ranges valid


In [15]:

# Crop-year coverage and duplicate audit

# Make sure year is numeric.
labeled["year"] = pd.to_numeric(
    labeled["year"],
    errors="coerce"
).astype("Int64")

# Actual years/crops present in labelled data
actual_years = set(labeled["year"].dropna().astype(int).unique())
actual_crops = set(labeled["crop"].dropna().astype(str).unique())

print("Expected years:", sorted(EXPECTED_YEARS))
print("Actual labelled years:", sorted(actual_years))

print("\nExpected crops:", sorted(EXPECTED_CROPS))
print("Actual labelled crops:", sorted(actual_crops))

# Create coverage table and FORCE the complete expected grid.
# This prevents KeyError when a year/crop has zero labelled rows.
coverage = (
    labeled
    .groupby(["year", "crop"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=sorted(EXPECTED_YEARS),
        columns=sorted(EXPECTED_CROPS),
        fill_value=0
    )
)

coverage.index.name = "year"
coverage.columns.name = "crop"

print("\nCrop-year labelled coverage:")
display(coverage)

# Find every missing crop-year combination safely
missing_combinations = [
    (int(year), crop)
    for year in coverage.index
    for crop in coverage.columns
    if coverage.loc[year, crop] == 0
]

print("\nTotal expected combinations:", len(EXPECTED_YEARS) * len(EXPECTED_CROPS))
print(
    "Combinations with at least one labelled row:",
    int((coverage > 0).sum().sum())
)
print("Zero-labelled combinations:", len(missing_combinations))

if missing_combinations:
    print("\nWARNING — crop-year combinations with zero labelled rows:")
    print(missing_combinations)
else:
    print("\nPASS: every expected crop-year combination has labelled rows.")

# Duplicate check
# Only use CORE_KEY columns that actually exist.
missing_core_key_cols = [
    col for col in CORE_KEY
    if col not in labeled.columns
]

if missing_core_key_cols:
    print(
        "\nWARNING — these CORE_KEY columns are not present:",
        missing_core_key_cols
    )

    duplicate_key = [
        col for col in ["state", "district", "crop", "year"]
        if col in labeled.columns
    ]

    print("Using fallback duplicate key:", duplicate_key)
else:
    duplicate_key = CORE_KEY

duplicate_count = labeled.duplicated(
    subset=duplicate_key
).sum()

print("\nDuplicate records using key:", duplicate_key)
print("Duplicate count:", duplicate_count)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate records using key {duplicate_key}"
)

print("PASS: no duplicate records detected.")

# MSP / FRP consistency checks
required_price_cols = ["crop", "support_price_type"]

missing_price_cols = [
    col for col in required_price_cols
    if col not in labeled.columns
]

if missing_price_cols:
    raise ValueError(
        f"Missing required support-price columns: {missing_price_cols}"
    )

invalid_price_types = (
    ~labeled["support_price_type"].isin(["MSP", "FRP"])
).sum()

print("\nInvalid support price types:", invalid_price_types)

assert invalid_price_types == 0, (
    "Unexpected values found in support_price_type."
)

# Sugarcane must use FRP
wrong_sugarcane = labeled.loc[
    (labeled["crop"] == "Sugarcane") &
    (labeled["support_price_type"] != "FRP")
]

# All other project crops must use MSP
wrong_other_crops = labeled.loc[
    (labeled["crop"] != "Sugarcane") &
    (labeled["support_price_type"] != "MSP")
]

print("Sugarcane rows with incorrect price type:", len(wrong_sugarcane))
print("Non-sugarcane rows with incorrect price type:", len(wrong_other_crops))

assert len(wrong_sugarcane) == 0, (
    "Some Sugarcane rows are not assigned FRP."
)

assert len(wrong_other_crops) == 0, (
    "Some non-Sugarcane rows are not assigned MSP."
)

print("\nPASS: MSP/FRP assignment is consistent.")
print("PASS: crop-year coverage and duplicate audit completed.")

Expected years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Actual labelled years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Expected crops: ['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Maize', 'Ragi', 'Rice', 'Soyabean', 'Sugarcane', 'Urad', 'Wheat']
Actual labelled crops: ['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Maize', 'Ragi', 'Rice', 'Soyabean', 'Sugarcane', 'Urad', 'Wheat']

Crop-year labelled coverage:


crop,Arhar/Tur,Bajra,Gram,Groundnut,Jowar,Maize,Ragi,Rice,Soyabean,Sugarcane,Urad,Wheat
year,,,,,,,,,,,,
2013,0,0,0,0,0,0,0,0,0,0,0,0
2014,436,264,423,366,289,552,179,583,214,473,473,485
2015,448,266,450,384,285,581,179,606,221,472,494,508
2016,471,267,458,380,280,577,180,604,229,464,509,504
2017,522,284,487,430,328,607,170,638,266,483,561,516
2018,526,319,488,469,354,609,177,644,282,517,566,511
2019,514,339,483,469,342,623,193,649,279,510,563,512
2020,519,333,504,467,355,613,199,648,287,482,566,524
2021,525,334,503,460,363,638,210,643,292,501,549,534



Total expected combinations: 144
Combinations with at least one labelled row: 132
Zero-labelled combinations: 12

WARNING — crop-year combinations with zero labelled rows:
[(2013, 'Arhar/Tur'), (2013, 'Bajra'), (2013, 'Gram'), (2013, 'Groundnut'), (2013, 'Jowar'), (2013, 'Maize'), (2013, 'Ragi'), (2013, 'Rice'), (2013, 'Soyabean'), (2013, 'Sugarcane'), (2013, 'Urad'), (2013, 'Wheat')]

Duplicate records using key: ['state', 'district', 'crop', 'soil_type', 'year']
Duplicate count: 0
PASS: no duplicate records detected.

Invalid support price types: 0
Sugarcane rows with incorrect price type: 0
Non-sugarcane rows with incorrect price type: 0

PASS: MSP/FRP assignment is consistent.
PASS: crop-year coverage and duplicate audit completed.


In [16]:
#Diagnostic outlier audit
rows = []
for col in numeric_features:
    s = pd.to_numeric(labeled[col], errors="coerce")
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((s < lo) | (s > hi)).sum())
    rows.append({
        "feature": col, "min": s.min(), "median": s.median(), "max": s.max(),
        "iqr_outlier_rows": n,
        "iqr_outlier_percent": round(n / len(labeled) * 100, 3)
    })
distribution_audit = pd.DataFrame(rows).sort_values("iqr_outlier_percent", ascending=False)
display(distribution_audit)


,feature,min,median,max,iqr_outlier_rows,iqr_outlier_percent
9,clayey_skeletal_fraction,0.000000,0.000000,0.834887,13282,22.827
4,historical_baseline_start_year,2013.000000,2013.000000,2023.000000,11650,20.022
11,sandy_fraction,0.000000,0.000000,0.823088,10945,18.811
1,area_ha,0.000000,1339.000000,997356.000000,9758,16.771
2,historical_10yr_baseline_yield_kg_ha,17.000000,1383.166667,220618.000000,6065,10.424
6,historical_baseline_yield_q_acre,0.420079,34.178793,5451.589495,6065,10.424
13,annual_mean_temp_c,-5.464615,25.663077,29.076154,5634,9.683
17,annual_wind_speed_m_s,0.910769,2.970769,5.874615,5615,9.650
15,annual_min_temp_c,-17.346154,15.762308,27.394615,5512,9.473
14,annual_max_temp_c,5.504615,36.082308,40.341538,4152,7.136


In [17]:
#Chronological train / validation / test split
train = labeled[labeled["year"].between(2013, 2022)].copy()
validation = labeled[labeled["year"] == 2023].copy()
test = labeled[labeled["year"] == 2024].copy()

assert len(train) > 0 and len(validation) > 0 and len(test) > 0
assert train["year"].max() < validation["year"].min() < test["year"].min()

def key_set(frame):
    return set(map(tuple, frame[CORE_KEY].to_numpy()))

assert key_set(train).isdisjoint(key_set(validation))
assert key_set(train).isdisjoint(key_set(test))
assert key_set(validation).isdisjoint(key_set(test))

split_summary = pd.DataFrame([
    ["train", "2013-2022", len(train), int((train[TARGET] == 0).sum()), int((train[TARGET] == 1).sum()), train[TARGET].mean() * 100],
    ["validation", "2023", len(validation), int((validation[TARGET] == 0).sum()), int((validation[TARGET] == 1).sum()), validation[TARGET].mean() * 100],
    ["test", "2024", len(test), int((test[TARGET] == 0).sum()), int((test[TARGET] == 1).sum()), test[TARGET].mean() * 100],
], columns=["split", "years", "rows", "success_0", "success_1", "success_rate_percent"])

split_summary["success_rate_percent"] = split_summary["success_rate_percent"].round(3)
display(split_summary)
assert len(train) + len(validation) + len(test) == len(labeled)


,split,years,rows,success_0,success_1,success_rate_percent
0,train,2013-2022,47345,18368,28977,61.204
1,validation,2023,5611,2132,3479,62.003
2,test,2024,5229,1807,3422,65.443


In [18]:
#Build the final handoff tables
metadata_columns = [
    c for c in [
        "soil_data_status", "climate_data_status", "environmental_data_status"
    ] if c in labeled.columns
]

handoff_columns = list(dict.fromkeys(CORE_KEY + FEATURES + [TARGET] + metadata_columns))

model_ready = labeled[handoff_columns].sort_values(CORE_KEY).reset_index(drop=True)
model_unlabeled = unlabeled[[c for c in handoff_columns if c != TARGET]].copy()
model_unlabeled = model_unlabeled.sort_values(
    [c for c in CORE_KEY if c in model_unlabeled.columns]
).reset_index(drop=True)

assert len(model_ready) == len(labeled)
assert model_ready[FEATURES].isna().sum().sum() == 0
assert model_ready[TARGET].isna().sum() == 0
assert model_ready.duplicated(CORE_KEY).sum() == 0

print("Final labelled handoff:", model_ready.shape)
print("Unlabelled retained:", model_unlabeled.shape)


Final labelled handoff: (58185, 36)
Unlabelled retained: (9641, 35)


In [19]:
#Feature dictionary
descriptions = {
    "year": "Agricultural year represented by starting year.",
    "state": "State name.",
    "district": "District name.",
    "crop": "Crop category.",
    "season": "Agricultural season/category.",
    "soil_type": "Dominant soil texture class.",
    "area_ha": "Cultivated area in hectares.",
    "historical_10yr_baseline_yield_kg_ha": "Mean yield from previous up to 10 agricultural years for State + District + Crop + Soil Type.",
    "historical_baseline_year_count": "Number of historical years contributing to baseline.",
    "historical_baseline_start_year": "Earliest year contributing to baseline.",
    "historical_baseline_end_year": "Latest year contributing to baseline.",
    "historical_baseline_yield_q_acre": "Historical baseline yield in quintals per acre.",
    "support_price_rs_per_quintal": "Government support-price benchmark in rupees per quintal.",
    "support_price_type": "MSP for non-sugarcane crops; FRP for sugarcane.",
    "clayey_fraction": "Clayey soil fraction.",
    "clayey_skeletal_fraction": "Clayey skeletal soil fraction.",
    "loamy_fraction": "Loamy soil fraction.",
    "sandy_fraction": "Sandy soil fraction.",
    "annual_rainfall_mm": "Annual rainfall from NASA POWER.",
    "annual_mean_temp_c": "Annual mean 2 m temperature in degrees Celsius.",
    "annual_max_temp_c": "Annual maximum 2 m temperature in degrees Celsius.",
    "annual_min_temp_c": "Annual minimum 2 m temperature in degrees Celsius.",
    "annual_relative_humidity_pct": "Annual relative humidity percentage.",
    "annual_wind_speed_m_s": "Annual 10 m wind speed in metres per second.",
    "annual_solar_radiation": "Annual NASA POWER surface solar radiation measure.",
    "monsoon_rainfall_mm": "Monsoon-period rainfall in millimetres.",
    "monsoon_mean_temp_c": "Monsoon-period mean temperature.",
    "monsoon_max_temp_c": "Monsoon-period maximum temperature.",
    "monsoon_min_temp_c": "Monsoon-period minimum temperature.",
    "monsoon_relative_humidity_pct": "Monsoon-period relative humidity.",
    "monsoon_wind_speed_m_s": "Monsoon-period 10 m wind speed.",
    "monsoon_solar_radiation": "Monsoon-period NASA POWER surface solar radiation measure.",
    "success_label": "Binary target: 1 when actual revenue meets/exceeds historical target revenue; 0 otherwise."
}

feature_dictionary = pd.DataFrame([
    {
        "feature": c,
        "role": "target" if c == TARGET else "feature",
        "feature_group": "categorical" if c in categorical_features else ("numeric" if c in numeric_features else "target"),
        "data_type": str(model_ready[c].dtype),
        "missing_rows": int(model_ready[c].isna().sum()),
        "unique_values": int(model_ready[c].nunique(dropna=True)),
        "description": descriptions.get(c, "")
    }
    for c in categorical_features + numeric_features + [TARGET]
])
display(feature_dictionary)


,feature,role,feature_group,data_type,missing_rows,unique_values,description
0,state,feature,categorical,str,0,34,State name.
1,district,feature,categorical,str,0,745,District name.
2,crop,feature,categorical,str,0,12,Crop category.
3,season,feature,categorical,str,0,1,Agricultural season/category.
4,soil_type,feature,categorical,str,0,4,Dominant soil texture class.
5,support_price_type,feature,categorical,str,0,2,MSP for non-sugarcane crops; FRP for sugarcane.
6,year,feature,numeric,Int64,0,11,Agricultural year represented by starting year.
7,area_ha,feature,numeric,float64,0,24336,Cultivated area in hectares.
8,historical_10yr_baseline_yield_kg_ha,feature,numeric,float64,0,35221,Mean yield from previous up to 10 agricultural...
9,historical_baseline_year_count,feature,numeric,int64,0,10,Number of historical years contributing to bas...


In [20]:
#Final quality audit and save
quality_audit = pd.DataFrame([
    ["source_rows", len(df)],
    ["labeled_rows", len(model_ready)],
    ["unlabeled_rows", len(model_unlabeled)],
    ["model_feature_count", len(FEATURES)],
    ["categorical_feature_count", len(categorical_features)],
    ["numeric_feature_count", len(numeric_features)],
    ["model_feature_missing_cells", int(model_ready[FEATURES].isna().sum().sum())],
    ["target_missing_cells", int(model_ready[TARGET].isna().sum())],
    ["duplicate_core_records", int(model_ready.duplicated(CORE_KEY).sum())],
    ["success_0_rows", int((model_ready[TARGET] == 0).sum())],
    ["success_1_rows", int((model_ready[TARGET] == 1).sum())],
    ["train_rows", len(train)],
    ["validation_rows", len(validation)],
    ["test_rows", len(test)],
    ["train_year_min", int(train.year.min())],
    ["train_year_max", int(train.year.max())],
    ["validation_year", 2023],
    ["test_year", 2024],
    ["irrigation_type_available", int("irrigation_type" in df.columns)]
], columns=["metric", "value"])
display(quality_audit)

# Final hard gates
assert len(df) == 67826
assert len(model_ready) == 58185
assert len(model_unlabeled) == 9641
assert model_ready[FEATURES].isna().sum().sum() == 0
assert model_ready[TARGET].isna().sum() == 0
assert model_ready.duplicated(CORE_KEY).sum() == 0

# Save datasets
model_ready.to_csv(LABELED_FILE, index=False)
model_unlabeled.to_csv(UNLABELED_FILE, index=False)

train[handoff_columns].sort_values(CORE_KEY).to_csv(TRAIN_FILE, index=False)
validation[handoff_columns].sort_values(CORE_KEY).to_csv(VALIDATION_FILE, index=False)
test[handoff_columns].sort_values(CORE_KEY).to_csv(TEST_FILE, index=False)

feature_dictionary.to_csv(FEATURE_DICT_FILE, index=False)
quality_audit.to_csv(AUDIT_FILE, index=False)

summary = {
    "project": "AgriRisk and ROI Prediction",
    "source_rows": int(len(df)),
    "labeled_rows": int(len(model_ready)),
    "unlabeled_rows": int(len(model_unlabeled)),
    "feature_count": int(len(FEATURES)),
    "target": TARGET,
    "success_0": int((model_ready[TARGET] == 0).sum()),
    "success_1": int((model_ready[TARGET] == 1).sum()),
    "success_rate_percent": round(float(model_ready[TARGET].mean() * 100), 4),
    "split": {"train": "2013-2022", "validation": "2023", "test": "2024"},
    "model_feature_missing_cells": int(model_ready[FEATURES].isna().sum().sum()),
    "duplicate_core_records": int(model_ready.duplicated(CORE_KEY).sum()),
    "irrigation_type_available": bool("irrigation_type" in df.columns),
    "leakage_exclusions": LEAKAGE_EXCLUSIONS
}
JSON_SUMMARY_FILE.write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary_text = (
    "AGRICRISK AND ROI PREDICTION - FINAL ML DATASET HANDOFF\n\n"
    f"Source rows: {len(df):,}\n"
    f"Labelled supervised rows: {len(model_ready):,}\n"
    f"Unlabelled retained rows: {len(model_unlabeled):,}\n"
    f"Model feature count: {len(FEATURES)}\n"
    f"Success 0: {(model_ready[TARGET] == 0).sum():,}\n"
    f"Success 1: {(model_ready[TARGET] == 1).sum():,}\n"
    f"Success rate: {model_ready[TARGET].mean()*100:.2f}%\n\n"
    "Chronological split:\n"
    f"Train 2013-2022: {len(train):,}\n"
    f"Validation 2023: {len(validation):,}\n"
    f"Test 2024: {len(test):,}\n\n"
    "Quality gates:\n"
    f"Missing model-feature cells: {model_ready[FEATURES].isna().sum().sum()}\n"
    f"Duplicate core records: {model_ready.duplicated(CORE_KEY).sum()}\n"
    f"Irrigation feature available: {'YES' if 'irrigation_type' in df.columns else 'NO'}\n\n"
    "Leakage exclusions:\n" + "\n".join("- " + x for x in LEAKAGE_EXCLUSIONS) + "\n\n"
    "Training rule: fit encoding, imputation, scaling and feature selection on training data only."
)
SUMMARY_FILE.write_text(summary_text, encoding="utf-8")

print(summary_text)
print("\nSaved files:")
for p in [LABELED_FILE, UNLABELED_FILE, TRAIN_FILE, VALIDATION_FILE, TEST_FILE,
          FEATURE_DICT_FILE, AUDIT_FILE, SUMMARY_FILE, JSON_SUMMARY_FILE]:
    print(p, "->", p.exists())


,metric,value
0,source_rows,67826
1,labeled_rows,58185
2,unlabeled_rows,9641
3,model_feature_count,32
4,categorical_feature_count,6
5,numeric_feature_count,26
6,model_feature_missing_cells,0
7,target_missing_cells,0
8,duplicate_core_records,0
9,success_0_rows,22307


AGRICRISK AND ROI PREDICTION - FINAL ML DATASET HANDOFF

Source rows: 67,826
Labelled supervised rows: 58,185
Unlabelled retained rows: 9,641
Model feature count: 32
Success 0: 22,307
Success 1: 35,878
Success rate: 61.66%

Chronological split:
Train 2013-2022: 47,345
Validation 2023: 5,611
Test 2024: 5,229

Quality gates:
Missing model-feature cells: 0
Duplicate core records: 0
Irrigation feature available: NO

Leakage exclusions:
- yield_kg_ha
- yield_q_acre
- production_tonnes
- actual_revenue_rs_per_acre
- revenue_vs_target_ratio
- yield_vs_historical_baseline_ratio
- target_revenue_rs_per_acre
- success_label

Training rule: fit encoding, imputation, scaling and feature selection on training data only.

Saved files:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\modeling\model_ready_labeled_2013_2025.csv -> True
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\modeling\model_unlabeled_2013_2025.csv -> True